# 10 — Four-Corpus Transfer Matrix and Equal-Label-Budget Study

Unified apparatus over four NF-v2 corpora (NF-CSE-CIC-IDS2018-v2, NF-UNSW-NB15-v2, NF-ToN-IoT-v2, NF-BoT-IoT-v2; identical 41-feature schema after identifier removal), giving 12 directed transfer pairs. Every training set is capped at 250k flows (stratified by attack family) and every evaluation set at 200k, applied identically to source fits, augment and the in-domain diagonal, so that all cells share one configuration. The in-domain diagonal serves as the oracle for the budget study (a model trained on the capped target training set is exactly the in-domain cell).

Budgets are 0.01%, 0.1%, 1%, 5% and 10% of the full target training partition (absolute sizes recorded as n_train). Strategies: zero_shot, calibrate (Platt on the buffer, frozen source model), buffer_only, augment (capped source + buffer; reference seed, three smallest budgets). Every cell records MCC at the 0.5 threshold and best-threshold MCC over a 199-point quantile grid, so ranking quality is separable from threshold placement. Seeds 42-44 for the matrix, zero_shot, calibrate and buffer_only; seed 42 for augment and bootstrap. Results append incrementally with resume-skip on (seed, source, target, model, budget, strategy).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'
os.makedirs(RESULT, exist_ok=True)

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seeds         = [42, 43, 44],
    ref_seed      = 42,
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    budgets       = [0.0001, 0.001, 0.01, 0.05, 0.10],
    aug_budgets   = [0.0001, 0.001, 0.01],
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
    boot_B        = 1000,
    boot_B_auprc  = 200,
)
MODELS = ['rf', 'lgbm', 'mlp']
MAIN_CSV = f'{RESULT}/fc_results.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train',
        'macro_f1', 'weighted_f1', 'mcc', 'auprc_macro', 'fp_rate', 'brier', 'ece',
        'mcc_best_thr', 'thr_best', 'single_class_buffer', 'fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
for tag, d in DATASETS.items():
    assert [c for c in d.columns if c not in ('Label', 'Attack')] == FEATURES, tag
    print(f'{tag}: {d.shape[0]:,} rows, attack rate {d.Label.mean():.4f}')
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(MAIN_CSV, mode='a', index=False,
                                          header=not os.path.exists(MAIN_CSV))

done = set()
if os.path.exists(MAIN_CSV):
    prev = pd.read_csv(MAIN_CSV)
    done = set(map(tuple, prev[['seed', 'source', 'target', 'model', 'budget', 'strategy']]
                   .astype(str).values))
    print(f'resume: {len(done)} rows already recorded')

def key(seed, src, tgt, m, b, s):
    return (str(seed), src, tgt, m, str(b), s)

def is_done(*k):
    return key(*k) in done

def mark(seed, src, tgt, m, b, s, metrics, n_train, single=0, fit_s=np.nan):
    row = dict(seed=seed, source=src, target=tgt, model=m, budget=b, strategy=s,
               n_train=n_train, single_class_buffer=single, fit_s=fit_s, **metrics)
    record(row)
    done.add(key(seed, src, tgt, m, b, s))
    print(f"  s{seed} {src}->{tgt} {m} b={b} {s}: MCC={metrics['mcc']:.3f} "
          f"bestthr={metrics['mcc_best_thr']:.3f} AUPRC={metrics['auprc_macro']:.3f} ECE={metrics['ece']:.3f}")

def fit_eval(mname, seed, Xtr, ytr, Xev, yev):
    model = make_model(mname, seed, len(Xtr))
    t0 = time.time()
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xev)[:, 1]
    m = all_metrics(yev, p)
    fit_s = round(time.time() - t0)
    del model
    gc.collect()
    return m, fit_s

boot_store = {}

for seed in CFG['seeds']:
    parts = {}
    for tag, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
        parts[tag] = dict(train_full=tr.reset_index(drop=True),
                          train=stratified_cap(tr, CFG['train_cap'], seed),
                          eval=stratified_cap(te, CFG['eval_cap'], seed))
    for src in CFG['corpora']:
        Xs, med_s = clean_X(parts[src]['train'], FEATURES)
        ys = parts[src]['train']['Label'].values
        for mname in MODELS:
            others = [t for t in CFG['corpora'] if t != src]
            need_source = seed == CFG['ref_seed'] or any(
                not is_done(seed, src, tgt, mname, 0, 'in_domain' if tgt == src else 'zero_shot')
                for tgt in CFG['corpora']) or any(
                not is_done(seed, src, tgt, mname, b, 'calibrate')
                for tgt in others for b in CFG['budgets'])
            need_other = any(not is_done(seed, src, tgt, mname, b, 'buffer_only')
                             for tgt in others for b in CFG['budgets']) or (
                seed == CFG['ref_seed'] and any(not is_done(seed, src, tgt, mname, b, 'augment')
                                                 for tgt in others for b in CFG['aug_budgets']))
            if not (need_source or need_other):
                continue
            model, fit_s = None, np.nan
            if need_source:
                model = make_model(mname, seed, len(Xs))
                t0 = time.time()
                model.fit(Xs, ys)
                fit_s = round(time.time() - t0)
                print(f'seed {seed} | source fit {mname} on {src} ({len(Xs):,} rows): {fit_s}s')
            for tgt in CFG['corpora']:
                ev = parts[tgt]['eval']
                yev = ev['Label'].values
                p = None
                if model is not None:
                    Xev, _ = clean_X(ev, FEATURES, medians=med_s)
                    p = model.predict_proba(Xev)[:, 1]
                    del Xev
                    strat = 'in_domain' if tgt == src else 'zero_shot'
                    if not is_done(seed, src, tgt, mname, 0, strat):
                        mark(seed, src, tgt, mname, 0, strat, all_metrics(yev, p), len(Xs), fit_s=fit_s)
                    if seed == CFG['ref_seed']:
                        boot_store[(src, mname, tgt)] = (yev.copy(), p.copy())
                if tgt == src:
                    continue
                for b in CFG['budgets']:
                    buf = stratified_frac(parts[tgt]['train_full'], b, seed)
                    ybuf = buf['Label'].values
                    single = int(len(np.unique(ybuf)) < 2)
                    if model is not None and not is_done(seed, src, tgt, mname, b, 'calibrate'):
                        Xb, _ = clean_X(buf, FEATURES, medians=med_s)
                        lr = platt_fit(model.predict_proba(Xb)[:, 1], ybuf)
                        mark(seed, src, tgt, mname, b, 'calibrate',
                             all_metrics(yev, platt_apply(lr, p, ybuf)), len(buf), single=single)
                        del Xb
                    if not is_done(seed, src, tgt, mname, b, 'buffer_only'):
                        Xbuf, mb = clean_X(buf, FEATURES)
                        Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
                        m, fs = fit_eval(mname, seed, Xbuf, ybuf, Xev_b, yev)
                        mark(seed, src, tgt, mname, b, 'buffer_only', m, len(buf), single=single, fit_s=fs)
                        del Xbuf, Xev_b
                        gc.collect()
                    if seed == CFG['ref_seed'] and b in CFG['aug_budgets'] and \
                       not is_done(seed, src, tgt, mname, b, 'augment'):
                        both = pd.concat([parts[src]['train'], buf], ignore_index=True)
                        Xa, ma = clean_X(both, FEATURES)
                        ya = both['Label'].values
                        Xev_a, _ = clean_X(ev, FEATURES, medians=ma)
                        m, fs = fit_eval(mname, seed, Xa, ya, Xev_a, yev)
                        mark(seed, src, tgt, mname, b, 'augment', m, len(both), single=single, fit_s=fs)
                        del both, Xa, Xev_a
                        gc.collect()
            if model is not None:
                del model
            gc.collect()
        del Xs
        gc.collect()

print('rows recorded:', len(done))

In [ ]:
def bootstrap_cis(y_true, p_pos, B, B_ap, rng):
    y = np.asarray(y_true); p = np.asarray(p_pos); n = len(y)
    keys = ['macro_f1', 'mcc', 'fp_rate', 'brier', 'ece']
    samples = {k: np.empty(B) for k in keys}
    samples['auprc_macro'] = np.empty(B_ap)
    for b in range(B):
        ix = rng.integers(0, n, n)
        yb, pb = y[ix], p[ix]
        pred = (pb >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yb, pred, labels=[0, 1]).ravel()
        samples['macro_f1'][b] = f1_score(yb, pred, average='macro')
        samples['mcc'][b] = matthews_corrcoef(yb, pred)
        samples['fp_rate'][b] = fp / (fp + tn) if (fp + tn) else np.nan
        samples['brier'][b] = np.mean((pb - yb) ** 2)
        samples['ece'][b] = ece_score(yb, pb, CFG['ece_bins'])
        if b < B_ap:
            samples['auprc_macro'][b] = (average_precision_score(yb, pb) +
                                         average_precision_score(1 - yb, 1 - pb)) / 2
    out = {}
    for k, arr in samples.items():
        out[f'{k}_lo'] = float(np.percentile(arr, 2.5))
        out[f'{k}_hi'] = float(np.percentile(arr, 97.5))
    return out

BOOT_CSV = f'{RESULT}/fc_matrix_bootstrap.csv'
rng = np.random.default_rng(CFG['ref_seed'])
rows = []
for (src, mname, tgt), (y, p) in sorted(boot_store.items()):
    t0 = time.time()
    ci = bootstrap_cis(y, p, CFG['boot_B'], CFG['boot_B_auprc'], rng)
    ci.update(source=src, model=mname, target=tgt, seed=CFG['ref_seed'],
              B=CFG['boot_B'], B_auprc=CFG['boot_B_auprc'])
    rows.append(ci)
    print(f"{src}/{mname}->{tgt}: {time.time()-t0:.0f}s  MCC [{ci['mcc_lo']:.3f}, {ci['mcc_hi']:.3f}]")
pd.DataFrame(rows).round(4).to_csv(BOOT_CSV, index=False)
print('saved', BOOT_CSV)

In [ ]:
from scipy.stats import spearmanr, wilcoxon

df = pd.read_csv(MAIN_CSV).drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])
metrics = ['mcc', 'mcc_best_thr', 'macro_f1', 'auprc_macro', 'fp_rate', 'ece']

# matrix aggregate over seeds
mat = df[df.strategy.isin(['in_domain', 'zero_shot'])]
agg = mat.groupby(['source', 'model', 'target', 'strategy'])[metrics].agg(['mean', 'std']).round(4)
agg.columns = ['_'.join(c) for c in agg.columns]
agg.reset_index().to_csv(f'{RESULT}/fc_matrix_aggregate.csv', index=False)

# budget aggregate over seeds
bud = df[df.strategy.isin(['zero_shot', 'calibrate', 'buffer_only', 'augment'])]
bagg = bud.groupby(['source', 'target', 'model', 'budget', 'strategy'])[metrics + ['n_train']].agg(['mean', 'std']).round(4)
bagg.columns = ['_'.join(c) for c in bagg.columns]
bagg.reset_index().to_csv(f'{RESULT}/fc_budget_aggregate.csv', index=False)

# diagnostic: zero-shot ranking quality vs post-calibration outcome, per (seed, pair, model)
z = df[df.strategy == 'zero_shot'][['seed', 'source', 'target', 'model', 'auprc_macro', 'mcc_best_thr', 'mcc']]
z = z.rename(columns={'auprc_macro': 'zs_auprc', 'mcc_best_thr': 'zs_best_thr', 'mcc': 'zs_mcc'})
c = df[df.strategy == 'calibrate'][['seed', 'source', 'target', 'model', 'budget', 'mcc', 'mcc_best_thr', 'ece']]
c = c.rename(columns={'mcc': 'cal_mcc', 'mcc_best_thr': 'cal_best_thr', 'ece': 'cal_ece'})
bo = df[df.strategy == 'buffer_only'][['seed', 'source', 'target', 'model', 'budget', 'mcc']].rename(columns={'mcc': 'bo_mcc'})
diag = z.merge(c, on=['seed', 'source', 'target', 'model']).merge(bo, on=['seed', 'source', 'target', 'model', 'budget'])
diag['external'] = 0
diag.to_csv(f'{RESULT}/fc_diagnostic.csv', index=False)

for bud_ in CFG['budgets']:
    sub = diag[diag.budget == bud_]
    rho, pv = spearmanr(sub.zs_auprc, sub.cal_best_thr)
    print(f'budget {bud_}: n={len(sub)}  Spearman(zero-shot AUPRC, calibrated best-thr MCC) = {rho:.3f} (p={pv:.2e})')

# buffer_only vs augment, paired, reference seed, augment budgets
pa = df[(df.seed == CFG['ref_seed']) & (df.strategy.isin(['buffer_only', 'augment'])) & (df.budget.isin(CFG['aug_budgets']))]
pv_ = pa.pivot_table(index=['source', 'target', 'model', 'budget'], columns='strategy', values='mcc').dropna()
w = wilcoxon(pv_.buffer_only, pv_.augment)
print(f'\nbuffer_only vs augment: n={len(pv_)} pairs, buffer_only>=augment in {(pv_.buffer_only >= pv_.augment).sum()}, '
      f'mean diff {(pv_.buffer_only - pv_.augment).mean():+.4f}, Wilcoxon p={w.pvalue:.3g}')
for b in CFG['aug_budgets']:
    s = pv_.xs(b, level='budget')
    print(f'  budget {b}: buffer_only mean {s.buffer_only.mean():.3f}, augment mean {s.augment.mean():.3f}')

print('\nMCC matrix (mean over seeds):')
for m in MODELS:
    print(f'--- {m} ---')
    display(mat[mat.model == m].pivot_table(index='source', columns='target', values='mcc', aggfunc='mean').round(3))
print('\ncalibrate MCC@0.5 by target (should be majority-class degenerate):')
display(df[df.strategy == 'calibrate'].groupby('target')[['mcc', 'fp_rate', 'mcc_best_thr']].agg(['mean', 'min', 'max']).round(3))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "10: four-corpus transfer matrix and equal-label-budget study (12 pairs, 3 models, seeds 42-44, threshold sweep, bootstrap, diagnostics)"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)